# On the very first run, the pipeline performs a FULL load; all subsequent runs switch to INCREMENTAL mode.

In [0]:
import requests
import time
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict
from pyspark.sql import Row
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [0]:
# ================= SECRETS =================
password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"
BATCH_SIZE    = 10000

ENDPOINTS = [
    {
        "name": "incidents",
        "url": "https://elixir.msf.org/tas/api/incidents",
        "fields": None,
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_incidents_raw",
        "pagination": "OFFSET"
    },

    {
        "name": "time_registrations",
        "url": "https://elixir.msf.org/tas/api/incidents/timeregistrations",
        "fields":"timeSpent,operator.id,creationDate,parent.id,notes,modificationDate",
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_timeregistrations_raw",
        "pagination": "NEXT"
         
    },

    {
        "name": "priorities",
        "url": "https://elixir.msf.org/tas/api/incidents/priorities",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_priorities_raw",
        "pagination": "SINGLE"

    },

    {
        "name": "statuses",
        "url": "https://elixir.msf.org/tas/api/incidents/statuses",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_statuses_raw",
        "pagination": "SINGLE"

    },

    {
        "name": "urgencies",
        "url": "https://elixir.msf.org/tas/api/incidents/urgencies",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_urgenciess_raw",
        "pagination": "SINGLE"
    },
    
    {
        "name": "categories",
        "url": "https://elixir.msf.org/tas/api/incidents/categories",
        "fields":None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_categories_raw",
        "pagination": "SINGLE"
    },
     
    {
        "name": "subcategories",
        "url": "https://elixir.msf.org/tas/api/incidents/subcategories",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_subcategories_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "closure_codes",
        "url": "https://elixir.msf.org/tas/api/incidents/closure_codes",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_closurecodes_raw",
        "pagination": "SINGLE"
    },
    

    {
        "name": "branches",
        "url": "https://elixir.msf.org/tas/api/branches",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_branches_raw",
        "pagination": "SINGLE"
        
    },

    {
        "name": "budgetholders",
        "url": "https://elixir.msf.org/tas/api/budgetholders",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_budgetholders_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "operators",
        "url": "https://elixir.msf.org/tas/api/operators",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_operators_raw",
        "pagination": "OFFSET"
    },
    
    {
        "name": "operatorgroup",
        "url": "https://elixir.msf.org/tas/api/operatorgroups",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_operatorgroup_raw",
        "pagination": "OFFSET"
    },

    {
        "name": "impacts",
        "url": "https://elixir.msf.org/tas/api/changes/impacts",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_changes_impacts_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "operatorchanges",
        "url": "https://elixir.msf.org/tas/api/operatorChanges",
        "fields": "id,incident.id,changeType,status,category.id,category.name,subcategory.id,processingStatus,modificationDate",
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_operator_changes_raw",
        "pagination": "NEXT"
    },

    {
        "name": "changeactivities",
        "url": "https://elixir.msf.org/tas/api/operatorChangeActivities",
        "fields": "timeSpent,id,category.id,lastModificationDate",
        "incremental_field": "lastModificationDate",
        "bronze_table": "elixir.brz.topdesk_operator_changeActivities_raw",
        "pagination": "NEXT"
    }
 ]




AUTH = (username, password)

PAGE_SIZE = 100
#MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30

LOAD_TYPE = "INCREMENTAL"   # "FULL" or "INCREMENTAL"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}.pipeline_state"

PIPELINE_NAME = "topdesk_ingestion"


In [0]:
def create_session() -> requests.Session:
    session = requests.Session()
    session.auth = AUTH
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    return session


def format_ts(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f") + "Z"


def parse_ts(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")
    return datetime.fromisoformat(value)

In [0]:
# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run(endpoint_name: str):
    if LOAD_TYPE == "FULL":
        return None
    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        AND endpoint = '{endpoint_name}'
        LIMIT 1
    """).collect()
    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)


def update_last_run(endpoint_name: str, ts: str):
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (
            SELECT '{PIPELINE_NAME}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================

def build_query(last_run, endpoint):
    inc_field = endpoint.get("incremental_field")
    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None
    return f"{inc_field}>'{last_run}'"


# ============================================================
# PARAMS
# ============================================================

def build_params(endpoint: dict, query: Optional[str], extra: Optional[dict] = None) -> dict:
    params = {}
    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]
    if query:
        params["query"] = query
    if extra:
        params.update(extra)
    return params

In [0]:
# ============================================================
# INGESTION
# ============================================================

def stream_endpoint_to_bronze(endpoint, query):
    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")
    pagination_type = endpoint["pagination"]

    if pagination_type not in {"SINGLE", "OFFSET", "NEXT"}:
        raise Exception(f"Invalid pagination type: {pagination_type}")

    print(f"[INFO] {endpoint_name} pagination: {pagination_type}")

    total = 0
    batch = []
    max_ts = None

    def track_ts(rec):
        nonlocal max_ts
        if inc_field:
            value = rec.get(inc_field)
            if value:
                parsed = parse_ts(value)
                if parsed and (max_ts is None or parsed > max_ts):
                    max_ts = parsed

    with create_session() as session:

        # =====================================================
        # SINGLE (NO PAGINATION)
        # =====================================================
        if pagination_type == "SINGLE":
            params = build_params(endpoint, query)
            resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()

            payload = resp.json()
            if isinstance(payload, dict):
                payload = payload.get("results", [])

            for rec in payload:
                track_ts(rec)
                batch.append(rec)
                total += 1

        # =====================================================
        # NEXT PAGINATION
        # =====================================================
        elif pagination_type == "NEXT":
            params = build_params(endpoint, query)
            url = endpoint["url"]

            while url:
                resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                payload = resp.json()
                for rec in payload.get("results", []):
                    track_ts(rec)
                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                print(f"[INFO] {endpoint_name} fetched={total}")

                url = payload.get("next")
                params = {}

        # =====================================================
        # OFFSET PAGINATION
        # =====================================================
        else:
            next_start = 0

            while True:
                params = build_params(endpoint, query, extra={"start": next_start, "page_size": PAGE_SIZE})
                resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                if not resp.text.strip():
                    break

                data = resp.json()
                if not data:
                    break

                for rec in data:
                    track_ts(rec)
                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < PAGE_SIZE:
                    break

                next_start += PAGE_SIZE
                print(f"[INFO] {endpoint_name} fetched={total}")

    if batch:
        write_to_bronze(batch, endpoint)

    return total, format_ts(max_ts) if max_ts else None


def write_to_bronze(records: List[Dict], endpoint) -> None:
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(raw_json=json.dumps(rec), ingestion_time=now)
        for rec in records
    ]

    df = spark.createDataFrame(rows)
    df.write.format("delta").mode("append").saveAsTable(bronze_table)

    print(f"[INFO] Written {len(records)} records to {bronze_table}")



In [0]:
# ============================================================
# MAIN
# ============================================================

def main():
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")
    ensure_state_table_exists()

    for endpoint in ENDPOINTS:
        print("=" * 60)
        endpoint_name = endpoint["name"]
        print(f"[PIPELINE] {endpoint_name}")

        last_run = get_last_run(endpoint_name)
        query = build_query(last_run, endpoint)

        if query:
            print(f"[INFO] Filter: {query}")
        else:
            print("[INFO] FULL load (no filter)")

        total, max_ts = stream_endpoint_to_bronze(endpoint, query)
        print(f"[DONE] {endpoint_name} -> {total} records")

        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts)
            print(f"[INFO] Updated last_run → {max_ts}")

In [0]:
if __name__ == "__main__":
    main()

## The codes below are just for testing the APIs before implementations

In [0]:

url = "https://elixir.msf.org/tas/api/operatorChanges"
#url = "https://elixir.msf.org/tas/api/incidents/closure_codes"

with requests.Session() as s:
    s.auth = AUTH
    r = s.get(url)
    print(r.status_code)
    #print(r.text)
    #print(r.json())

In [0]:
url = "https://elixir.msf.org/tas/api/incidents/closure_codes"
with requests.Session() as session:
    session.auth = AUTH

    r1 = session.get(
        url,
        params={"start": 0, "page_size": 1},
        timeout=REQUEST_TIMEOUT
    )
    r1.raise_for_status()
    data1 = r1.json()

    r2 = session.get(
        url,
        params={"start": 1, "page_size": 100},
        timeout=REQUEST_TIMEOUT
    )
    r2.raise_for_status()
    data2 = r2.json()

print("page_size=1 ->", len(data1))
print("page_size=100 ->", len(data2))
print("First IDs equal?:", data1[0]["id"] == data2[0]["id"])